In [105]:
%pip install --upgrade google-adk google-cloud-aiplatform litellm requests ipdb --quiet

In [106]:
import os
from typing import Dict, List, Optional
from IPython.display import display, Markdown
import requests
import vertexai
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from vertexai.preview import reasoning_engines

# Pull secrets and config from the environment — never hardcode these.
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")  # required by LiteLLM for Claude
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")

for name, value in [
    ("GOOGLE_MAPS_API_KEY", GOOGLE_MAPS_API_KEY),
    ("ANTHROPIC_API_KEY", ANTHROPIC_API_KEY),
    ("GOOGLE_CLOUD_PROJECT", PROJECT_ID),
]:
    if not value:
        print(f"WARNING: {name} is not set in the environment.")

vertexai.init(project=PROJECT_ID, location=LOCATION)

In [107]:
def get_lat_lon(location: str) -> Optional[Dict[str, float]]:
    """
    Convert a place name or address into latitude and longitude using the
    Google Maps Geocoding API.

    Args:
        location (str): A place name, city, or address (e.g., "College Station, TX").

    Returns:
        Optional[Dict[str, float]]: A dictionary with 'lat' and 'lon' keys.
        Returns None if the location cannot be found or an error occurs.
    """
    # breakpoint()
    if not GOOGLE_MAPS_API_KEY:
        return "google api key not found"

    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": location, "key": GOOGLE_MAPS_API_KEY}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("status") != "OK" or not data.get("results"):
            return None

        coords = data["results"][0]["geometry"]["location"]
        return {"lat": coords["lat"], "lon": coords["lng"]}
    except (requests.RequestException, KeyError, IndexError):
        return "exception, try again"

In [108]:
print(get_lat_lon("College Station, TX"))

{'lat': 30.6210482, 'lon': -96.3255016}


In [109]:
def get_google_weather_forecast(
    lat: float, lon: float
) -> Optional[List[Dict[str, str]]]:
    """
    Fetch the extended daily weather forecast from the Google Maps Platform
    Weather API based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of daily forecast dictionaries,
        each with 'date', 'daytimeForecast', 'nighttimeForecast', 'maxTemp',
        and 'minTemp'. Returns None if data is unavailable or an error occurs.
    """
    if not GOOGLE_MAPS_API_KEY:
        return None

    url = "https://weather.googleapis.com/v1/forecast/days:lookup"
    params = {
        "key": GOOGLE_MAPS_API_KEY,
        "location.latitude": lat,
        "location.longitude": lon,
        "unitsSystem": "IMPERIAL"
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        days = data.get("forecastDays", [])

        return [
            {
                "date": (
                    f"{day['displayDate']['year']}-"
                    f"{day['displayDate']['month']:02d}-"
                    f"{day['displayDate']['day']:02d}"
                ),
                "daytimeForecast": day.get("daytimeForecast", {})
                    .get("weatherCondition", {})
                    .get("description", {})
                    .get("text", "N/A"),
                "nighttimeForecast": day.get("nighttimeForecast", {})
                    .get("weatherCondition", {})
                    .get("description", {})
                    .get("text", "N/A"),
                "maxTemp": f"{day.get('maxTemperature', {}).get('degrees', 'N/A')}°"
                    f"{day.get('maxTemperature', {}).get('unit', '')}",
                "minTemp": f"{day.get('minTemperature', {}).get('degrees', 'N/A')}°"
                    f"{day.get('minTemperature', {}).get('unit', '')}",
            }
            for day in days
        ]
    except (requests.RequestException, KeyError):
        return None

In [110]:
def get_gov_weather_forecast(
    lat: float, lon: float
) -> Optional[List[Dict[str, str]]]:
    """
    Fetch the extended weather forecast from the U.S. National Weather Service
    API based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast period dictionaries,
        each with 'name', 'temperature', 'shortForecast', and 'detailedForecast'.
        Returns None if data is unavailable (including for non-US locations,
        which NWS does not cover) or an error occurs.
    """
    # NWS requires a descriptive User-Agent identifying the application.
    headers = {"User-Agent": "(google-lab-readynow-weather-agent)"}

    try:
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        points_resp = requests.get(points_url, headers=headers, timeout=10)
        points_resp.raise_for_status()
        forecast_url = points_resp.json()["properties"]["forecast"]

        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_resp.raise_for_status()
        periods = forecast_resp.json()["properties"]["periods"]

        return [
            {
                "name": period["name"],
                "temperature": f"{period['temperature']}°{period['temperatureUnit']}",
                "shortForecast": period["shortForecast"],
                "detailedForecast": period["detailedForecast"],
            }
            for period in periods
        ]
    except (requests.RequestException, KeyError):
        return None

In [112]:
location = "College Station, TX"
lat_lon = get_lat_lon(location)
# print("Google forecast")
# get_google_weather_forecast(lat_lon['lat'],lat_lon['lon'])
print("NWS Forecast")
get_gov_weather_forecast(lat_lon['lat'],lat_lon['lon'])

NWS Forecast


[{'name': 'Today',
  'temperature': '102°F',
  'shortForecast': 'Sunny',
  'detailedForecast': 'Sunny, with a high near 102. Heat index values as high as 103. South wind 5 to 10 mph.'},
 {'name': 'Tonight',
  'temperature': '78°F',
  'shortForecast': 'Mostly Clear',
  'detailedForecast': 'Mostly clear, with a low around 78. Heat index values as high as 103. South wind 5 to 10 mph, with gusts as high as 20 mph.'},
 {'name': 'Tuesday',
  'temperature': '101°F',
  'shortForecast': 'Mostly Sunny',
  'detailedForecast': 'Mostly sunny, with a high near 101. Heat index values as high as 104. Southwest wind 5 to 10 mph.'},
 {'name': 'Tuesday Night',
  'temperature': '78°F',
  'shortForecast': 'Mostly Clear',
  'detailedForecast': 'Mostly clear, with a low around 78. Heat index values as high as 104. South wind 5 to 10 mph.'},
 {'name': 'Wednesday',
  'temperature': '101°F',
  'shortForecast': 'Sunny',
  'detailedForecast': 'Sunny, with a high near 101. Southwest wind around 5 mph.'},
 {'name':

In [113]:
WEATHER_AGENT_INSTRUCTIONS = """
You are a helpful weather assistant covering locations in the United States.

When a user asks about weather for a location:
1. Call get_lat_lon to convert the location name into latitude/longitude.
2. Call get_extended_weather_forecast with those coordinates to retrieve
   the forecast from the National Weather Service.
3. Summarize the forecast in plain, easy-to-read language. If any period's
   forecast mentions severe weather (storms, extreme heat, flooding, winter
   weather, etc.), lead your response with a clear "ALERT:" line.

If get_lat_lon returns None, tell the user you couldn't find that location.
If get_extended_weather_forecast returns None, tell the user the forecast
is unavailable, and note that the National Weather Service only covers
US locations.
"""

In [114]:
weather_agent = Agent(
   name="marvin",
   model="gemini-2.5-flash",
   description=("Marvin the Friendly Weather Agent."),
   instruction=(WEATHER_AGENT_INSTRUCTIONS),
tools=[get_gov_weather_forecast, get_lat_lon]
)


In [115]:
from vertexai.preview import reasoning_engines
app = reasoning_engines.AdkApp(
   agent=weather_agent
)


In [116]:
test_user_id = "test-user-id"
session = app.create_session(user_id=test_user_id)
print(session)

{'id': '6143f795-9708-4aec-a4da-48823bff6179', 'app_name': 'default-app-name', 'user_id': 'test-user-id', 'state': {}, 'events': [], 'last_update_time': 1787589311.8008637}


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


In [123]:

cities = ['College Station, TX', 'Miami, FL']
for city in cities:
    display(Markdown(f"## Forecast for {city}"))
    for event in app.stream_query(
        user_id=test_user_id,
        session_id=session['id'],
        message=f"What's the weather like in {city} this week?",
    ):
        if 'finish_reason' in event.keys():
            try:
                display(Markdown(event['content']['parts'][0]['text'].replace('\n','  \n')))
            except:
                pass


## Forecast for College Station, TX

ALERT: Expect dangerously high heat index values throughout Tuesday, reaching as high as 104°F. There is also a chance of showers and thunderstorms on Thursday and Friday.  
  
Here's the extended weather forecast for College Station, TX:  
  
**Today and Tonight:** Sunny with a high near 102°F, feeling as hot as 103°F with the heat index. Tonight will be mostly clear with a low around 78°F, still feeling like 103°F due to the heat index. Winds will be from the south at 5 to 10 mph, with gusts up to 20 mph tonight.  
  
**Tuesday and Tuesday Night:** Mostly sunny with a high near 101°F, feeling as hot as 104°F with the heat index. Tuesday night will be mostly clear with a low around 78°F, with a heat index up to 104°F. Winds will be from the southwest at 5 to 10 mph.  
  
**Wednesday and Wednesday Night:** Sunny with a high near 101°F. Wednesday night will be mostly clear with a low around 79°F.  
  
**Thursday and Thursday Night:** Mostly sunny with a high near 100°F, with a 30% chance of showers and thunderstorms after 1 PM. Thursday night brings a 40% chance of showers and thunderstorms before 7 PM and again between 7 PM and 1 AM, with a low around 76°F.  
  
**Friday and Friday Night:** Mostly sunny with a high near 97°F, and a slight chance (20%) of showers and thunderstorms after 1 PM. Friday night also has a slight chance of showers and thunderstorms before 7 PM, then mostly clear with a low around 76°F.  
  
**Weekend (Saturday & Sunday):** Expect sunny days with highs near 97°F on Saturday and 96°F on Sunday. Nights will be mostly clear with lows around 75°F.

## Forecast for Miami, FL

ALERT: Expect dangerously high heat index values throughout the week, reaching as high as 105°F. There is also a continuous chance of showers and thunderstorms every day and night this week, with the highest probabilities on Wednesday.  
  
Here's the extended weather forecast for Miami, FL this week:  
  
**Today:** Mostly sunny with a high near 89°F, but feeling as hot as 105°F with the heat index. There's a 30% chance of showers and thunderstorms before noon, potentially bringing a tenth to a quarter of an inch of rain. The wind will be from the southeast at 3 to 9 mph.  
  
**Tonight:** Partly cloudy with a low around 83°F. The heat index will still be as high as 100°F.  
  
**Tuesday:** Sunny with a high near 90°F, feeling as hot as 105°F with the heat index. There's a slight chance (20%) of showers and thunderstorms after 3 PM.  
  
**Tuesday Night:** Partly cloudy with a low around 83°F. The heat index will be as high as 103°F, and there's a 50% chance of showers and thunderstorms.  
  
**Wednesday:** Mostly sunny with a high near 89°F. Showers and thunderstorms are likely, with a 70% chance of precipitation, particularly between 8 AM and 2 PM.  
  
**Wednesday Night:** Mostly cloudy with a low around 83°F and a 50% chance of showers and thunderstorms.  
  
**Thursday:** Mostly sunny with a high near 89°F and a 40% chance of showers and thunderstorms.  
  
**Thursday Night:** Partly cloudy with a low around 82°F and a 40% chance of showers and thunderstorms.  
  
**Friday:** Mostly sunny with a high near 90°F and a 40% chance of showers and thunderstorms.  
  
**Friday Night:** Partly cloudy with a low around 82°F and a 40% chance of showers and thunderstorms before 2 AM.  
  
**Weekend (Saturday & Sunday):** Mostly sunny with highs near 90°F both days. There's a 50% chance of showers and thunderstorms each day. Nights will be partly cloudy with lows around 82°F, and a 50% chance of showers and thunderstorms.